In [ ]:
!pip install mcp google-auth-oauthlib nest_asyncio


In [1]:
import asyncio
import nest_asyncio

from google_auth_oauthlib.flow import InstalledAppFlow

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

nest_asyncio.apply()

In [2]:
SCOPES = [
    "https://www.googleapis.com/auth/gmail.readonly",
    "https://www.googleapis.com/auth/gmail.compose",
]

flow = InstalledAppFlow.from_client_secrets_file(
    "C:/Users/mohammed.musthaq/AiProjects/client_secret.json",
    scopes=SCOPES,
)

credentials = flow.run_local_server(port=8080)

print("Authenticated!")

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=964935368755-5h1sfu13rougqducv621kvc4qvlrsoh4.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.compose&state=ShnkqfBCnc8pRePtnDRgKV144Bwekd&code_challenge=0UmCIoXadTa2aEKtoraf4zYwgvvaGXA63RZ8F0dwTN0&code_challenge_method=S256&access_type=offline
Authenticated!


In [5]:
credentials.token


'ya29.a0AT3oNZ-AdyaI2_f3iutAVETmoMknMPwptLkQx01trDwVysnK1gN0Ecg01dg6fLOzwb-uq51YWa0gRNck0ilpUmKlUKiMR7yuEs3m0ZEfKpBt4Rq8P9r79VyzQB9Xog0JZTt7hW6aC1h1j2vqRaRSD3tdWCqHVirkyYaDH7zwmlMYPDMbGw8IoLh7cFHTrmqtRyW3qjgaCgYKAdwSARMSFQHGX2MirMRRcldT6xeiSi3o80Dybw0206'

In [ ]:
# import json
# cred_path = "C:/Users/mohammed.musthaq/AiProjects/client_secret.json"
# with open(cred_path, 'r') as f:
#     credentials = json.load(f)
# print(credentials['installed']['token_uri'])

In [6]:
import requests


r = requests.get(
    "https://gmail.googleapis.com/gmail/v1/users/me/profile",
    headers={
        "Authorization": f"Bearer {credentials.token}"
    }
)

print(r.status_code)
print(r.text)

200
{
  "emailAddress": "mushtaq.it.5037@gmail.com",
  "messagesTotal": 10099,
  "threadsTotal": 9906,
  "historyId": "1363461"
}



In [7]:
async def connect():

    async with streamablehttp_client(
        "https://gmailmcp.googleapis.com/mcp/v1",
        headers={
            "Authorization": f"Bearer {credentials.token}"
        },
    ) as (read_stream, write_stream, _):

        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:

            await session.initialize()

            print("Connected!")

            tools = await session.list_tools()

            return tools

In [8]:
tools = asyncio.get_event_loop().run_until_complete(connect())

for idx, tool in enumerate(tools.tools):
    # print("=" * 80)
    print(f"{idx}.{tool.name}")
    # print(tool.description)

Connected!
0.create_draft
1.list_drafts
2.get_thread
3.get_message
4.search_threads
5.label_thread
6.unlabel_thread
7.apply_sensitive_thread_label
8.list_labels
9.label_message
10.unlabel_message
11.apply_sensitive_message_label
12.create_label


In [9]:
import json

for tool in tools.tools:
    if tool.name == "search_threads":
        print(json.dumps(tool.inputSchema, indent=2))

{
  "description": "Request message for SearchThreads RPC.",
  "properties": {
    "includeTrash": {
      "description": "Optional. Include drafts from TRASH in the results. Defaults to false.",
      "type": "boolean"
    },
    "pageSize": {
      "description": "Optional. The maximum number of threads to return. If unspecified, defaults to 20. The maximum allowed value is 50.",
      "format": "int32",
      "type": "integer"
    },
    "pageToken": {
      "description": "Optional. Page token to retrieve a specific page of results in the list. Leave empty to fetch the first page. This is primarily used for pagination to continue fetching results from where the previous `SearchThreads` call left off, especially when the number of threads matching the query exceeds the page_size limit.",
      "type": "string"
    },
    "query": {
      "description": "Optional. A query string to filter the threads. Natural language queries must be pre-converted into Gmail syntax queries to use thi

In [10]:
async def search():

    async with streamablehttp_client(
        "https://gmailmcp.googleapis.com/mcp/v1",
        headers={
            "Authorization": f"Bearer {credentials.token}"
        },
    ) as (read_stream, write_stream, _):

        async with ClientSession(read_stream, write_stream) as session:

            await session.initialize()

            result = await session.call_tool(
                "search_threads",
                {
                    # "query": "from:jobalerts-noreply@linkedin.com newer_than:1d"
                    # "query": "newer_than:1d"
                    "query":""
                },
            )
            

            return result
        
result = asyncio.get_event_loop().run_until_complete(search())

result

CallToolResult(meta=None, content=[TextContent(type='text', text='The caller does not have permission', annotations=None, meta=None)], structuredContent=None, isError=True)

In [12]:
!pip install google-api-python-client google-auth google-auth-oauthlib google-auth-httplib2

  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/15.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.6 MB ? eta -:--:--
    --------------------------------------- 0.3/15.6 MB ? eta -:--:--
   -- ------------------------------------- 0.8/15.6 MB 2.6 MB/s eta 0:00:06
   ------ --------------------------------- 2.6/15.6 MB 5.4 MB/s eta 0:00:03
   ------------------ --------------------- 7.1/15.6 MB 10.4 MB/s eta 0:00:01
   ------------------------------ --------- 12.1/15.6 MB 13.7 MB/s eta 0:00:01
   --------------------------------- ------ 13.1/15.6 MB 14.2 MB/s eta 0:00:01
   --------------------------------- ------ 13.1/15.6 MB 14.2 MB/s eta 0:00:01
   --------------------------------- ------ 13.1/15.6 MB 14.2 MB/s eta 0:00:01
   --------------------------------- ------ 13.1/15.6 MB 14.2 MB/s eta 0:00:01
   --------------------------------- ------ 13.1/15.6 MB 14.2 MB/s eta 0:00:01
   ---------------

In [13]:

from googleapiclient.discovery import build

service = build("gmail", "v1", credentials=credentials)

def search_threads(query: str = "", max_results: int = 10):
    """
    Search Gmail threads.

    Args:
        query: Gmail search query
        max_results: Maximum number of threads to return

    Returns:
        List of thread IDs
    """

    response = (
        service.users()
        .threads()
        .list(
            userId="me",
            q=query,
            maxResults=max_results
        )
        .execute()
    )

    return response.get("threads", [])

def get_thread(thread_id: str):
    """
    Retrieve complete thread.
    """

    return (
        service.users()
        .threads()
        .get(
            userId="me",
            id=thread_id,
            format="full"
        )
        .execute()
    )

threads = search_threads(
    query="from:jobalerts-noreply@linkedin.com newer_than:3d",
    max_results=5
)

threads

[{'id': '19f455ef9ea2e4c9',
  'snippet': 'ADASI Senior AI Engineer: External Job DescriptionJob Title: Senior AI… ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏',
  'historyId': '1363102'},
 {'id': '19f40389b1d7e14d',
  'snippet': 'ADASI Senior AI Engineer: External Job DescriptionJob Title: Senior AI… ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏',
  'historyId': '1362937'},
 {'id': '19f3e8102f946b57',
  'snippet': 'See your latest job matches ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏',
  'historyId': '1362687'},
 {'id': '19f3e1328ee6e748',
  'snippet': 'See your latest job matches ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏

In [16]:
thread = get_thread(threads[0]["id"])

def print_thread_summary(thread):

    print("=" * 80)

    for message in thread["messages"]:

        headers = {
            h["name"]: h["value"]
            for h in message["payload"]["headers"]
        }

        print("Subject :", headers.get("Subject"))
        print("From    :", headers.get("From"))
        print("Date    :", headers.get("Date"))
        print("Snippet :", message.get("snippet"))
        print("-" * 80)

print_thread_summary(thread)

Subject : Senior AI Engineer at ADASI
From    : LinkedIn Job Alerts <jobalerts-noreply@linkedin.com>
Date    : Thu, 9 Jul 2026 05:34:36 +0000 (UTC)
Snippet : ADASI Senior AI Engineer: External Job DescriptionJob Title: Senior AI… ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏
--------------------------------------------------------------------------------


In [17]:
import base64


def get_email_body(message):

    payload = message["payload"]

    if "parts" in payload:
        for part in payload["parts"]:
            if part["mimeType"] == "text/plain":
                data = part["body"]["data"]
                return base64.urlsafe_b64decode(data).decode("utf-8")

    if "body" in payload and "data" in payload["body"]:
        data = payload["body"]["data"]
        return base64.urlsafe_b64decode(data).decode("utf-8")

    return ""

In [18]:
body = get_email_body(thread["messages"][-1])

print(body)

Your job alert for ai engineer in United Arab Emirates          
New jobs match your preferences.
Manage alerts: https://www.linkedin.com/comm/jobs/alerts?lipi=urn%3Ali%3Apage%3Aemail_email_job_alert_digest_01%3BuK43ZSrFRsya3%2BpFeEvrqQ%3D%3D&midToken=AQGyVTUALXtnIA&midSig=3X9XSP230KBIk1&trk=eml-email_job_alert_digest_01-primary_job_list-0-manage_alerts_text_ssid_5393990273_fmid_b7bbg1~mrd2nxix~5o&trkEmail=eml-email_job_alert_digest_01-primary_job_list-0-manage_alerts_text_ssid_5393990273_fmid_b7bbg1~mrd2nxix~5o-null-b7bbg1~mrd2nxix~5o-null-null&eid=b7bbg1-mrd2nxix-5o&otpToken=MTQwMjE5ZTQxMzJkY2JjZGIzMjQwNGVkNGUxZGUyYjM4YWNjZDc0Nzk4YWM4ZDYxNzBjZjA2NmI0NzVjNWVmNWYwZDJkZmU1NjRlNmI4Zjk1YWEzZThmYTAwZjE3NWRjMmY2ODg5YjBiNjc2YWU4MmI0MzMyYywxLDE%3D
            
Senior AI Engineer
ADASI
Abu Dhabi
View job: https://www.linkedin.com/comm/jobs/view/4408647846/?trackingId=Wl3S6A2LXysJWKuWeyqrjg%3D%3D&refId=WlYPDZ9W6531mca15Ek63g%3D%3D&lipi=urn%3Ali%3Apage%3Aemail_email_job_alert_digest_01%3BuK43ZSr